In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import mlflow
import joblib

In [2]:
def score_model(model, X, y):
    y_pred = model.predict(X)
    scores = {
            'precision': precision_score(y, y_pred),
            'recall': recall_score(y, y_pred),
            'f1_score': f1_score(y, y_pred)
    }
    return scores

def evaluate_predictions(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    
    return cm, report


def create_model(model_type, params):
    if model_type == 'logistic_regression':
        classifier = LogisticRegression(**params)
    elif model_type == 'naive_bayes':
        classifier = MultinomialNB(**params)
    elif model_type == 'random_forest':
        classifier = RandomForestClassifier(**params)

    model = Pipeline([('vectorizer', TfidfVectorizer(stop_words = 'english')),
                      ('model', classifier)])
    return model

def train(experiment: dict):
    model_type = experiment['model_type']
    params = experiment["params"]

    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("SpamFilter")
    mlflow.autolog()
# Load datasets
    train = pd.read_csv('../data/train.csv')
    validation = pd.read_csv('../data/validation.csv')

    # Split datasets
    X_train, y_train = train['message'], train['label']
    X_validation, y_validation = validation['message'], validation['label']

    # Create model
    model = create_model(model_type, params)

    # Train model
    with mlflow.start_run(run_name=f"{model_type}_experiment"):

        mlflow.log_param("model_type", model_type)
        mlflow.log_params(params)

        model.fit(X_train, y_train)
        y_validation_pred = model.predict(X_validation)
        y_validation_scores = model.predict_proba(X_validation)[:, 1]


        cm, report = evaluate_predictions(y_validation, y_validation_pred)
        scores = score_model(model, X_validation, y_validation)
        aucpr = average_precision_score(y_validation, y_validation_scores)
        scores["aucpr"] = aucpr

        mlflow.log_metrics(scores)
        mlflow.log_text(str(cm), "confusion_matrix.txt")
        mlflow.log_text(report, "classification_report.txt")
        model_info = mlflow.sklearn.log_model(model, "model")

        mlflow.register_model(model_uri=model_info.model_uri, name="SpamFilterClassifier") 

### Logistic Regression Experiment

In [3]:
logistic_regression_exp = {
    "model_type": "logistic_regression",
    "params": {
        "max_iter": 1000
    }
}

train(logistic_regression_exp)

2026/03/09 20:45:11 INFO mlflow.tracking.fluent: Experiment with name 'SpamFilter' does not exist. Creating a new experiment.
2026/03/09 20:45:13 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/09 20:45:14 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/09 20:45:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/09 20:45:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/09 20:45:36 WARNING mlflow.sklearn: Unrecognized dataset type

🏃 View run logistic_regression_experiment at: http://127.0.0.1:5000/#/experiments/1/runs/10a858550c6b41f89f43d9565031761f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Naive Bayes Experiment

In [4]:
naive_bayes_exp = {
    "model_type": "naive_bayes",
    "params": {}
}
train(naive_bayes_exp)

2026/03/09 20:46:19 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/09 20:46:20 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/09 20:46:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/09 20:46:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/09 20:46:34 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/09 20:46:36 WARNING mlflow.sklearn: Unrecognized datase

🏃 View run naive_bayes_experiment at: http://127.0.0.1:5000/#/experiments/1/runs/f63e58ad988946ff82137134c4467563
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Random Forest Experiment

In [5]:
random_forest_exp = {
    "model_type": "random_forest",
    "params": {
        "n_estimators": 100,
        "random_state": 42
    }
}
train(random_forest_exp)

2026/03/09 20:46:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/09 20:46:50 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/09 20:46:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/09 20:46:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/09 20:46:59 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/09 20:47:02 WARNING mlflow.sklearn: Unrecognized datase

🏃 View run random_forest_experiment at: http://127.0.0.1:5000/#/experiments/1/runs/123bc9b57f8046af9062e1a73ca42839
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### AUCPR For Three Benchmark models

In [6]:
test = pd.read_csv("../data/test.csv")

X_test = test["message"]
y_test = test["label"]

In [8]:
model_v1 = mlflow.sklearn.load_model("models:/SpamFilterClassifier/1")
model_v2 = mlflow.sklearn.load_model("models:/SpamFilterClassifier/2")
model_v3 = mlflow.sklearn.load_model("models:/SpamFilterClassifier/3")

In [9]:
probs_v1 = model_v1.predict_proba(X_test)[:, 1]
probs_v2 = model_v2.predict_proba(X_test)[:, 1]
probs_v3 = model_v3.predict_proba(X_test)[:, 1]

2026/03/09 20:48:06 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/09 20:48:06 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/03/09 20:48:07 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.


In [10]:
aucpr_v1 = average_precision_score(y_test, probs_v1)
aucpr_v2 = average_precision_score(y_test, probs_v2)
aucpr_v3 = average_precision_score(y_test, probs_v3)

print("Model 1 AUCPR:", aucpr_v1)
print("Model 2 AUCPR:", aucpr_v2)
print("Model 3 AUCPR:", aucpr_v3)

Model 1 AUCPR: 0.9871890592616896
Model 2 AUCPR: 0.9823224723212031
Model 3 AUCPR: 0.9889551541192376


In [12]:
joblib.dump(model_v3, "../models/best_model.pkl")

['../models/best_model.pkl']